# EO Explorer - offline walkthrough

This notebook runs the app's **pure core** with no network and no STAC catalogue. It validates a synthetic AOI, synthesises one scene of bands, computes NDVI / NDWI / NDMI with the same index functions the app uses, and shows a robust percentile stretch and a (guarded) colourised preview.

Everything here is seeded and reproducible - the same numbers come out of `make demo`.

## 1. Run the demo

`run_demo(0)` returns the headline metrics and writes `outputs/summary.json`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `from app import ...` resolves
from app import demo, stac, render
import numpy as np

metrics = demo.run_demo(seed=0, out_dir='../outputs')
metrics

## 2. AOI validation

The same gate the UI uses, on the synthetic AOI.

In [ ]:
bbox = demo.DEMO_BBOX
v = stac.validate_aoi(bbox)
print('ok:', v.ok)
print('area_km2:', round(v.area_km2, 3))
print(v.message)

## 3. Index statistics

NDVI / NDWI / NDMI summary stats over the synthetic scene, plus the geometry helpers (centre, aspect ratio, suggested zoom).

In [ ]:
bands = demo._synthesize_bands(0, demo.DEMO_SHAPE)
ndvi = render._ndvi(bands['nir'], bands['red'])
ndwi = render._ndwi(bands['green'], bands['nir'])
ndmi = render._ndmi(bands['nir'], bands['swir16'])
for name, arr in [('NDVI', ndvi), ('NDWI', ndwi), ('NDMI', ndmi)]:
    print(name, render.index_stats(arr))
print('center      ', stac.bbox_center(bbox))
print('aspect ratio', round(stac.bbox_aspect_ratio(bbox), 4))
print('suggest zoom', stac.suggest_zoom(bbox))

## 4. Robust percentile stretch

`percentile_stretch` gives contrast bounds that ignore outliers and NaN, which is what you want before colour-mapping.

In [ ]:
vmin, vmax = render.percentile_stretch(ndvi)
print('NDVI 2nd/98th percentile:', round(vmin, 4), round(vmax, 4))
counts, edges = render.histogram(ndvi, bins=8)
print('histogram counts:', counts.tolist())

## 5. Colourised preview (guarded)

If matplotlib and Pillow are installed, colourise NDVI and show it; otherwise skip cleanly.

In [ ]:
try:
    import matplotlib.pyplot as plt
    spec = render.INDEX_REGISTRY['NDVI']
    rgba = render.colorize(ndvi, vmin=spec.vmin, vmax=spec.vmax, colormap=spec.colormap)
    plt.figure(figsize=(4, 4))
    plt.imshow(rgba)
    plt.title('Synthetic NDVI preview')
    plt.axis('off')
    plt.show()
except Exception as exc:
    print('Preview skipped (matplotlib/PIL not available):', exc)